<a href="https://colab.research.google.com/github/hristinagjorgjievska/Natural-Language-Processing/blob/master/lab2/lab2_doma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install evaluate
!pip install bert_score

In [6]:
import pandas as pd

In [7]:
data = pd.read_csv('sarcasm.txt', sep='\t')
data = data.sample(frac=1, ignore_index=True)

In [8]:
data

,Style 1,Style 2
0,just stuck my entire hand in a hot oven withou...,Self preservation isnt my strong suit
1,Being a mom with an autoimmune disease means I...,I wouldn't because sarcasm is the only weapon ...
2,Keanu reeves is killing his role in ozark seas...,The brother in ozark looks just like Keanu ree...
3,"“tit for tat, butter for fat, you kill my dog,...",My ancestors spoke about a lot of true things
4,Whoop diddy scoop poop,I would’ve just tweeted that I hated Kanye’s n...
...,...,...
688,Searching for Pokemon later because I hate mys...,I am going to play Pokémon go because I love m...
689,im glad we finally have a party in opposition ...,"I would say something like ""it is supposed to ..."
690,i love jumping on zoom meetings where everyone...,I'm sick of listening to my coworkers talk abo...
691,@goldfishbabe101 crispy all day every day over...,"""dry but honestly not mad about it."""


In [9]:
train_size = int(0.8 * len(data))
val_size = int(0.1 * len(data))

In [10]:
train_data = data[:train_size]
val_data = data[train_size:train_size + val_size]
test_data = data[train_size + val_size:]

In [11]:
train_sarcasm = train_data['Style 1'].values.tolist()
train_sincere = train_data['Style 2'].values.tolist()

In [12]:
val_sarcasm = val_data['Style 1'].values.tolist()
val_sincere = val_data['Style 2'].values.tolist()

In [13]:
test_sarcasm = test_data['Style 1'].values.tolist()
test_sincere = test_data['Style 2'].values.tolist()

In [14]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [15]:
tokenizer = AutoTokenizer.from_pretrained('t5-small')

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [16]:
inputs = tokenizer(train_sarcasm, truncation=True)
outs = tokenizer(text_target=train_sincere, truncation=True)

In [17]:
from datasets import Dataset

In [18]:
train_dataset = Dataset.from_dict({'input_ids': inputs['input_ids'],
                                   'attention_mask': inputs['attention_mask'],
                                   'labels': outs['input_ids']})

In [19]:
from transformers import DataCollatorForSeq2Seq

In [20]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer)

In [27]:
def train(model, train_loader, optimizer, epochs=5, device='cuda'):
  model = model.to(device)
  model.train()

  for epoch in range(epochs):
    total_loss = 0.00

    for batch in train_loader:
      batch = {k: v.to(device) for k, v in batch.items()}
      outputs = model(**batch)
      loss = outputs.loss
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()
      total_loss += loss.item()

    avg_loss = total_loss / max(1, len(train_loader))
    print(f'Epoch {epoch + 1}/{epochs}, Loss: {avg_loss:.4f}')

In [28]:
from evaluate import load

In [29]:
bleu = load('bleu')
bertscore = load('bertscore')

In [33]:
def generate_and_score(model, tokenizer, prompts, targets):
  model = model.to('cuda').eval()
  predictions = []
  for prompt in prompts:
    tokens = tokenizer([prompt], return_tensors='pt').to('cuda')
    out = model.generate(**tokens, max_new_tokens=50)
    predictions.append(tokenizer.decode(out[0], skip_special_tokens=True))

  bl = bleu.compute(predictions=predictions, references=targets)['bleu']
  f1 = bertscore.compute(predictions=predictions, references=targets, model_type='distilroberta-base')['f1']

  return bl, sum(f1) / len(f1)

In [34]:
from torch.utils.data import DataLoader
from torch.optim import AdamW

best_score = 0

for lr in [0.0001, 0.0001]:
  for epochs in [2, 5]:
    for batch_size in [8, 16]:
      model = AutoModelForSeq2SeqLM.from_pretrained('t5-small')
      train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=data_collator)
      optimizer = AdamW(model.parameters(), lr=lr)
      train(model, train_loader, optimizer, epochs)

      bl, f1 = generate_and_score(model, tokenizer, val_sarcasm, val_sincere)

      print(lr, epochs, batch_size, 'BLEU: ', bl, "BERTScore: ", f1)

      if f1 > best_score:
        best_score = f1
        best_model = model

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Epoch 1/2, Loss: 3.3716
Epoch 2/2, Loss: 3.0624


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

... (16 outputs left out)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Epoch 1/5, Loss: 3.3396
Epoch 2/5, Loss: 3.0668
Epoch 3/5, Loss: 2.8914
Epoch 4/5, Loss: 2.7716
Epoch 5/5, Loss: 2.6723
0.0001 5 8 BLEU:  0.1190722649438034 BERTScore:  0.8998319610305454


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Epoch 1/5, Loss: 3.4018
Epoch 2/5, Loss: 3.1066
Epoch 3/5, Loss: 2.9874
Epoch 4/5, Loss: 2.8799
Epoch 5/5, Loss: 2.8074
0.0001 5 16 BLEU:  0.1177010444118331 BERTScore:  0.8972972579624342


# ОБЈАСНУВАЊЕ НА РЕЗУЛТАТИТЕ
- Секоја од 8те комбинации ја евалуирав на валидациското множество со BLEU и BERTScore. Како најдобра комбинација се покажа lr=0.0001, 5 epochs и batch 16, со BERTScore 0.8973 и BLEU 0.1177.
- Изборот го направив според резултатите од BERTScore-от, бидејќи неутралната реченица користи многу различен стил и зборови од саркастичната, па затоа BLEU што е метрика која бара точни совпаѓања има ниски вредности дури и кога значењето на речениците се совпаѓа.

In [35]:
generate_and_score(best_model, tokenizer, test_sarcasm, test_sincere)

(0.0901214309017329, 0.8961809039115906)

In [36]:
model_name = 't5-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [37]:
zero_shot = []
for sample in test_sarcasm:
    zero_shot.append(f'Rewrite the following sarcastic text as a neutral text: {sample}')

In [38]:
examples = f'Input: {train_sarcasm[0]} Output: {train_sincere[0]}\nInput: {train_sarcasm[1]} Output: {train_sincere[1]}\n'

few_shot = []
for sample in test_sarcasm:
    few_shot.append(f'Examples:\n{examples}Rewrite the following sarcastic text as a neutral text: {sample} Output: ')

In [39]:
generate_and_score(model, tokenizer, zero_shot, test_sincere)

(0.008056330718020373, 0.8355640547616141)

In [40]:
generate_and_score(model, tokenizer, few_shot, test_sincere)

(0.0, 0.8318410847868238)

# ОБЈАСНУВАЊЕ ЗА РЕЗУЛТАТИТЕ
- Најдобар се покажа T5-small моделот што го трениравме (BLEU = 0.0901, BERTScore 0.8962) бидејќи научил точно како саркастичната реченица да ја претвори во неутрална. T5-base без тренирање е многу полош и со zero-shot и со few-shot, бидејќи T5 не е научен да следи инструкции и не може да не разбере што бараме од него. Кај few shot не беа од помош примерите бидејќи тој не знае како да ги искористи.

In [48]:
data = pd.read_csv('sarcasm.txt', sep='\t')
data = data.sample(frac=1, random_state=42, ignore_index=True)

In [49]:
train_size = int(0.8 * len(data))
val_size = int(0.1 * len(data))

In [50]:
train_data = data[:train_size]
val_data = data[train_size:train_size + val_size]
test_data = data[train_size + val_size:]

In [51]:
train_texts = train_data['Style 1'].tolist() + train_data['Style 2'].tolist()
val_texts = val_data['Style 1'].tolist() + val_data['Style 2'].tolist()
test_texts = test_data['Style 1'].tolist() + test_data['Style 2'].tolist()

In [52]:
train_labels = [1] * len(train_data) + [0] * len(train_data)
val_labels = [1] * len(val_data) + [0] * len(val_data)
test_labels = [1] * len(test_data) + [0] * len(test_data)

In [53]:
class SarcasmDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [54]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [55]:
def print_metrics(labels, preds):
    print('accuracy:', accuracy_score(labels, preds))
    print('precision:', precision_score(labels, preds))
    print('recall:', recall_score(labels, preds))
    print('f1:', f1_score(labels, preds))

In [56]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

In [57]:
def train_model(model_name, lr, epochs, batch_size):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

    train_dataset = SarcasmDataset(tokenizer(train_texts, truncation=True), train_labels)
    val_dataset = SarcasmDataset(tokenizer(val_texts, truncation=True), val_labels)

    training_args = TrainingArguments(output_dir='out/', report_to='none', learning_rate=lr,
                                      num_train_epochs=epochs, per_device_train_batch_size=batch_size,
                                      eval_strategy='epoch', logging_strategy='epoch', save_strategy='no')

    trainer = Trainer(model=model, args=training_args,
                      train_dataset=train_dataset, eval_dataset=val_dataset,
                      processing_class=tokenizer)

    output = trainer.train()
    print(model_name, lr, epochs, batch_size, 'time:', output.metrics['train_runtime'])
    print_metrics(val_labels, trainer.predict(val_dataset).predictions.argmax(axis=1))
    return trainer, tokenizer

In [58]:
train_model('roberta-base', 0.00001, 5, 16)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

... (3 outputs left out)

Epoch,Training Loss,Validation Loss
1,0.608516,0.415396
2,0.310582,0.343014
3,0.220173,0.376132
4,0.156573,0.468035
5,0.122567,0.503608


roberta-base 1e-05 5 16 time: 60.3863


accuracy: 0.855072463768116
precision: 0.855072463768116
recall: 0.855072463768116
f1: 0.855072463768116


(<transformers.trainer.Trainer at 0x7c9263606f90>,
 RobertaTokenizer(name_or_path='roberta-base', vocab_size=50265, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}, added_tokens_decoder={
 	0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
 	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
 	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
 	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
 	50264: AddedToken("<mask>", rstrip=False, lstrip=True, single_word=False, normalized=True, special=True),
 }))

In [59]:
train_model('distilbert-base-uncased', 0.00001, 5, 16)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

... (2 outputs left out)

Epoch,Training Loss,Validation Loss
1,0.562054,0.484712
2,0.335633,0.454654
3,0.263830,0.450176
4,0.220075,0.451696
5,0.198150,0.459916


distilbert-base-uncased 1e-05 5 16 time: 28.1468


accuracy: 0.8260869565217391
precision: 0.8260869565217391
recall: 0.8260869565217391
f1: 0.8260869565217391


(<transformers.trainer.Trainer at 0x7c9263451450>,
 BertTokenizer(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
 	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 }))

In [60]:
best_name = 'roberta-base'

In [61]:
for lr in [0.0001, 0.00001]:
    for epochs in [2, 5]:
        for batch_size in [16, 32]:
            train_model(best_name, lr, epochs, batch_size)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,0.469091,0.473925
2,0.270427,0.458722


roberta-base 0.0001 2 16 time: 23.4166


accuracy: 0.8695652173913043
precision: 0.8493150684931506
recall: 0.8985507246376812
f1: 0.8732394366197183


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

... (30 outputs left out)

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,0.663678,0.602255
2,0.478232,0.394777
3,0.310182,0.407697
4,0.244382,0.367737
5,0.213405,0.363152


roberta-base 1e-05 5 32 time: 61.6248


accuracy: 0.8260869565217391
precision: 0.8461538461538461
recall: 0.7971014492753623
f1: 0.8208955223880597


In [62]:
trainer, tokenizer = train_model(best_name, 0.00001, 5, 16)
test_dataset = SarcasmDataset(tokenizer(test_texts, truncation=True), test_labels)
print_metrics(test_labels, trainer.predict(test_dataset).predictions.argmax(axis=1))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,0.598233,0.380098
2,0.297104,0.332439
3,0.201867,0.361979
4,0.143551,0.435747
5,0.103324,0.455379


roberta-base 1e-05 5 16 time: 59.2447


accuracy: 0.8405797101449275
precision: 0.8615384615384616
recall: 0.8115942028985508
f1: 0.835820895522388


accuracy: 0.9
precision: 0.9666666666666667
recall: 0.8285714285714286
f1: 0.8923076923076924


In [63]:
tokenizer = AutoTokenizer.from_pretrained('t5-small')
model = AutoModelForSeq2SeqLM.from_pretrained('t5-small')

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [64]:
def classify(prompt):
    tokens = tokenizer(prompt, return_tensors='pt')
    out = model.generate(**tokens, max_new_tokens=5)
    answer = tokenizer.decode(out[0], skip_special_tokens=True)
    return 1 if 'yes' in answer.lower() else 0

In [65]:
zero_shot_preds = []
for s in test_texts:
    zero_shot_preds.append(classify(f'Is the following text sarcastic? Answer yes or no: {s}'))

print_metrics(test_labels, zero_shot_preds)

accuracy: 0.5785714285714286
precision: 0.5533980582524272
recall: 0.8142857142857143
f1: 0.6589595375722543


In [66]:
examples = f'Text: {train_texts[0]} Answer: yes\nText: {train_texts[-1]} Answer: no\n'

few_shot_preds = []
for s in test_texts:
    few_shot_preds.append(classify(f'{examples}Is the following text sarcastic? Answer yes or no: {s} Answer: '))

print_metrics(test_labels, few_shot_preds)

accuracy: 0.4357142857142857
precision: 0.44
recall: 0.4714285714285714
f1: 0.45517241379310347


# Објаснување на резултатите
- Со prompting и T5-small резултатите се полоши:
  - zero-shot (acc: 0.58 и f1: 0.66)
  - few-shot (acc: 0.44 и f1: 0.46)

- Тоа е блиску до погодување случајно бидејќи е околу 0.5, спроти acc: 0.90 кај RoBERTa. Причината е што T5-small е мал модел и не е трениран да следи инструкции, па затоа и не разбира што го прашуваме. Примерите во few-shot не беа исто така добри бидејќи не му помогнаа.